# Section 4 · Notebook 1 — Supervisor–Worker Pattern

A market-research assistant:
- **News Agent** — recent news and developments (Tavily web search)
- **Financial Agent** — prices, history and fundamentals (Yahoo Finance via `yfinance`)
- **Sentiment Agent** — what investors are saying (Tavily web search)
- **Supervisor** — splits the question, delegates to the workers, and writes one recommendation

```
                    ┌──────────────┐
   question ──────► │  Supervisor  │ ──────► one combined answer
                    └──────┬───────┘
          ┌────────────────┼────────────────┐
          ▼                ▼                ▼
     News Agent     Financial Agent   Sentiment Agent
```

**You need** a free [Tavily](https://www.tavily.com/) API key. An [Alpha Vantage](https://www.alphavantage.co/support/#api-key) key is optional.

In [2]:
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
from strands.models.bedrock import BedrockModel

MODEL_ID = os.getenv("BEDROCK_MODEL_ID", "global.anthropic.claude-sonnet-4-6")
REGION = os.getenv("AWS_REGION", os.getenv("AWS_DEFAULT_REGION", "us-east-1"))
os.environ.setdefault("AWS_REGION", REGION)

model = BedrockModel(model_id=MODEL_ID, region_name=REGION)
print(f"Using {MODEL_ID} in {REGION}")

Using global.anthropic.claude-sonnet-4-6 in us-east-1


## Configure API keys

Put them in a `.env` file at the repo root (recommended — it is git-ignored), or set them in your shell before starting Jupyter:

```bash
export TAVILY_API_KEY="tvly-..."
export ALPHA_VANTAGE_API_KEY="..."   # optional
```

…or uncomment the lines below.

In [4]:
from dotenv import load_dotenv

load_dotenv()  # reads TAVILY_API_KEY (and ALPHA_VANTAGE_API_KEY) from a .env file, if present
# os.environ["TAVILY_API_KEY"] = "tvly-..."
# os.environ["ALPHA_VANTAGE_API_KEY"] = "..."

if not os.getenv("TAVILY_API_KEY"):
    print("⚠️  TAVILY_API_KEY not set — the news and sentiment agents cannot search the web.")
else:
    print("✓ TAVILY_API_KEY is configured")

if not os.getenv("ALPHA_VANTAGE_API_KEY"):
    print("ℹ️  ALPHA_VANTAGE_API_KEY not set — financial data comes from yfinance (free, no key).")
else:
    print("✓ ALPHA_VANTAGE_API_KEY is configured")

✓ TAVILY_API_KEY is configured
ℹ️  ALPHA_VANTAGE_API_KEY not set — financial data comes from yfinance (free, no key).


In [5]:
import yfinance as yf
from strands import Agent, tool
from strands_tools.tavily import tavily_search

## Custom financial tools (Yahoo Finance)

In [6]:
def _num(value, money=False, pct=False):
    """Format numbers safely — Yahoo sometimes leaves fields empty."""
    if not isinstance(value, (int, float)):
        return "N/A"
    if pct:
        return f"{value * 100:.2f}%"
    return f"${value:,.2f}" if money else f"{value:,.2f}"


@tool
def get_stock_price(symbol: str) -> str:
    """Get the current stock price and key metrics using Yahoo Finance.

    Args:
        symbol: Stock ticker symbol (e.g., 'AAPL', 'TSLA', 'MSFT')
    """
    try:
        info = yf.Ticker(symbol).info
        return (
            f"Stock: {symbol}\n"
            f"Current Price: {_num(info.get('currentPrice') or info.get('regularMarketPrice'), money=True)}\n"
            f"Market Cap: {_num(info.get('marketCap'), money=True)}\n"
            f"P/E Ratio: {_num(info.get('trailingPE'))}\n"
            f"52 Week High: {_num(info.get('fiftyTwoWeekHigh'), money=True)}\n"
            f"52 Week Low: {_num(info.get('fiftyTwoWeekLow'), money=True)}\n"
            f"Average Volume: {_num(info.get('averageVolume'))}\n"
            f"Sector: {info.get('sector', 'N/A')}\n"
            f"Industry: {info.get('industry', 'N/A')}"
        )
    except Exception as e:
        return f"Error fetching stock data for {symbol}: {e}"


@tool
def get_stock_history(symbol: str, period: str = "1mo") -> str:
    """Get historical stock price data.

    Args:
        symbol: Stock ticker symbol (e.g., 'AAPL', 'TSLA')
        period: One of 1d, 5d, 1mo, 3mo, 6mo, 1y, 2y, 5y, 10y, ytd, max
    """
    try:
        hist = yf.Ticker(symbol).history(period=period)
        if hist.empty:
            return f"No historical data available for {symbol}"
        start_price = hist["Close"].iloc[0]
        end_price = hist["Close"].iloc[-1]
        change_pct = (end_price - start_price) / start_price * 100
        return (
            f"Historical Data for {symbol} ({period}):\n"
            f"Starting Price: ${start_price:.2f}\n"
            f"Ending Price: ${end_price:.2f}\n"
            f"Change: {change_pct:+.2f}%\n"
            f"Highest: ${hist['High'].max():.2f}\n"
            f"Lowest: ${hist['Low'].min():.2f}\n"
            f"Average Volume: {hist['Volume'].mean():,.0f}\n\n"
            f"Recent 5-day trend:\n{hist[['Close', 'Volume']].tail().to_string()}"
        )
    except Exception as e:
        return f"Error fetching historical data for {symbol}: {e}"


@tool
def get_company_financials(symbol: str) -> str:
    """Get company financial metrics: revenue, margins, growth and leverage.

    Args:
        symbol: Stock ticker symbol (e.g., 'AAPL', 'TSLA')
    """
    try:
        info = yf.Ticker(symbol).info
        return (
            f"Financial Metrics for {symbol}:\n"
            f"Revenue: {_num(info.get('totalRevenue'), money=True)}\n"
            f"Revenue Growth: {_num(info.get('revenueGrowth'), pct=True)}\n"
            f"Gross Profit: {_num(info.get('grossProfits'), money=True)}\n"
            f"Profit Margin: {_num(info.get('profitMargins'), pct=True)}\n"
            f"Operating Margin: {_num(info.get('operatingMargins'), pct=True)}\n"
            f"EBITDA: {_num(info.get('ebitda'), money=True)}\n"
            f"Earnings Growth: {_num(info.get('earningsGrowth'), pct=True)}\n"
            f"Return on Equity: {_num(info.get('returnOnEquity'), pct=True)}\n"
            f"Debt to Equity: {_num(info.get('debtToEquity'))}\n"
            f"Free Cash Flow: {_num(info.get('freeCashflow'), money=True)}"
        )
    except Exception as e:
        return f"Error fetching financial data for {symbol}: {e}"


print(get_stock_price(symbol="AAPL"))  # tools are still plain functions you can call

Stock: AAPL
Current Price: $341.27
Market Cap: $4,980,555,644,928.00
P/E Ratio: 39.14
52 Week High: $348.15
52 Week Low: $243.42
Average Volume: 45,038,190.00
Sector: Technology
Industry: Consumer Electronics


## Optional: Alpha Vantage tool

Use this instead of (or alongside) yfinance if you have an Alpha Vantage key. Other providers worth knowing: Polygon.io, Finnhub, Financial Modeling Prep.

In [7]:
from alpha_vantage.timeseries import TimeSeries


@tool
def get_alpha_vantage_quote(symbol: str) -> str:
    """Get the latest stock quote from the Alpha Vantage API.

    Args:
        symbol: Stock ticker symbol (e.g., 'AAPL')
    """
    key = os.getenv("ALPHA_VANTAGE_API_KEY")
    if not key:
        return "ALPHA_VANTAGE_API_KEY is not set."
    ts = TimeSeries(key=key, output_format="pandas")
    data, _ = ts.get_quote_endpoint(symbol=symbol)
    return data.to_string()


financial_tools = [get_stock_price, get_stock_history, get_company_financials, tavily_search]
if os.getenv("ALPHA_VANTAGE_API_KEY"):
    financial_tools.append(get_alpha_vantage_quote)

## Worker agents

In [8]:
news_agent = Agent(
    name="news_agent",
    description="Researches recent news and developments about companies and industries.",
    model=model,
    system_prompt="""You are a news research specialist. Search for recent news and developments about companies and industries.

When searching:
- Focus on recent news (use time_range='week' or 'month')
- Prefer reliable sources such as major news outlets
- Summarize key developments and trends
- Include specific facts, dates, and figures when available""",
    tools=[tavily_search],
    callback_handler=None,
)

financial_agent = Agent(
    name="financial_agent",
    description="Analyzes stock prices, price history, and company financial metrics.",
    model=model,
    system_prompt="""You are a financial analysis specialist. Analyze financial metrics, stock performance, and company valuations.

When analyzing:
- Use get_stock_price for current prices and key metrics
- Use get_stock_history for price trends
- Use get_company_financials for revenue, earnings, and growth
- Compare companies within the same sector and give specific numbers""",
    tools=financial_tools,
    callback_handler=None,
)

sentiment_agent = Agent(
    name="sentiment_agent",
    description="Analyzes market and investor sentiment from forums, social media, and commentary.",
    model=model,
    system_prompt="""You are a market sentiment analyst. Analyze market sentiment from social media, forums, and investor discussions.

When analyzing sentiment:
- Look for both bullish and bearish opinions
- Identify common concerns and excitement points
- Distinguish retail from institutional sentiment
- Summarize the overall market mood""",
    tools=[tavily_search],
    callback_handler=None,
)
print("✓ Worker agents created")

✓ Worker agents created


## Supervisor agent

`agent.as_tool()` wraps a worker so the supervisor can call it like any other tool. (Passing the agents directly in `tools=[...]` does the same thing.)

In [9]:
supervisor = Agent(
    name="research_supervisor",
    model=model,
    system_prompt="""You are a market research coordinator. When asked about investment opportunities:

1. Work out what information is needed
2. Delegate to the specialist agents:
   - news_agent: recent developments and industry news
   - financial_agent: financial metrics and performance data
   - sentiment_agent: market sentiment and investor opinions
3. Synthesize their responses into one balanced recommendation

This is educational analysis, not financial advice — say so briefly at the end.""",
    tools=[news_agent.as_tool(), financial_agent.as_tool(), sentiment_agent.as_tool()],
)
print("✓ Supervisor created")

✓ Supervisor created


## Run the multi-agent system

In [10]:
question = "Should I invest in electric vehicle companies?"
print(f"Question: {question}\n")

response = supervisor(question)

Question: Should I invest in electric vehicle companies?

Great question! To give you a well-rounded analysis of the electric vehicle (EV) sector, I'll simultaneously consult all three of my specialist agents — covering recent news, financial metrics, and market sentiment. Let me gather that data now!
Tool #1: news_agent

Tool #2: financial_agent

Tool #3: sentiment_agent


DEPRECATION WARNING: tavily is deprecated. This warning becomes an error log in v0.9.0. Migration path: use the official Tavily MCP server, documented at https://docs.tavily.com/documentation/mcp.
DEPRECATION WARNING: tavily is deprecated. This warning becomes an error log in v0.9.0. Migration path: use the official Tavily MCP server, documented at https://docs.tavily.com/documentation/mcp.
DEPRECATION WARNING: tavily is deprecated. This warning becomes an error log in v0.9.0. Migration path: use the official Tavily MCP server, documented at https://docs.tavily.com/documentation/mcp.
DEPRECATION WARNING: tavily is deprecated. This warning becomes an error log in v0.9.0. Migration path: use the official Tavily MCP server, documented at https://docs.tavily.com/documentation/mcp.
DEPRECATION WARNING: tavily is deprecated. This warning becomes an error log in v0.9.0. Migration path: use the official Tavily MCP server, documented at https://docs.tavily.com/documentation/mcp.
DEPRECATION WAR

╭───────────────────────────────────────────── Tavily Search Results ─────────────────────────────────────────────╮
│ Query: Tesla latest news developments 2025                                                                      │
│                                                                                                                 │
│ Results: 4 found                                                                                                │
│ --------------------------------------------------                                                              │
│                                                                                                                 │
│ [1] Doubling Down | Serving Northern Nevada                                                                     │
│ URL: https://www.nnbw.com/news/2026/oct/13/doubling-down                                                        │
│ Score: 0.6410185                                                                                                │
│ Content: # Doubling Down                                                                                        │
│                                                                                                                 │
│ ### Manufacturing jobs in Northern Nevada have doubled in a decade                                              │
│                                                                                                                 │
│ Tesla in Storey County added about 1,000 additional jobs in 2025 to work on production at its new semi-truck    │
│ manufacturing facility adjacent to the Gigafactory.                                                             │
│                                                                                                                 │
│ Tesla in Storey County added about 1,000 additional jobs in 2025 to work on production at its new semi-truck    │
│ manufacturing facility adjacent to the Gigafactory.                                                             │
│ Courtesy Tesla                                                                                                  │
│                                                                                                                 │
│ By Rob Sabo                                                                                                     │
│                                                                                                                 │
│ Tuesday, October 13, 2026 [...] Dozens of blue-chip, industry-leading companies have established large-scale    │
│ manufacturing operations in the north state, though perhaps none is larger than Tesla Gigafactory Nevada, which │
│ in 2023 employed about 11,000 people. Tesla in 2025 added about 1,000 additional employees as it ramped up      │
│ production at its new semi-truck manufacturing facility adjacent to the Gigafactory.                            │
│                                                                                                                 │
│                                                                                                                 │
│ [2] Tesla (TSLA) Stock Price, Quote, Forecast & News | TECHi                                                    │
│ URL: https://www.techi.com/quote/TSLA                                                                           │
│ Score: 0.6213807                                                                                                │
│ Content: Earnings calendar and report history                                                                   │
│                                                                                                                 │
│ Countdown                                                                                                       │
│                                                       

╭───────────────────────────────────────────── Tavily Search Results ─────────────────────────────────────────────╮
│ Query: electric vehicle industry latest news 2025                                                               │
│                                                                                                                 │
│ Results: 5 found                                                                                                │
│ --------------------------------------------------                                                              │
│                                                                                                                 │
│ [1] U.S. EV Sales Rebound in Q3 2026                                                                            │
│ URL:                                                                                                            │
│ https://electriccarsreport.com/2026/10/u-s-ev-sales-rebound-in-q3-2026-as-toyota-rivian-and-subaru-gain-ground  │
│ Score: 0.8405867                                                                                                │
│ Content: Tesla continued to lead the American electric vehicle market by a wide margin, even as its sales       │
│ declined.                                                                                                       │
│                                                                                                                 │
│ The automaker sold 144,094 electric vehicles in Q3 2026, down 19.7% compared with the same quarter in 2025.     │
│ Despite the decline, Tesla accounted for approximately 54% of all U.S. EV sales during the period. [...]        │
│ Despite the quarterly improvement, EV sales remained significantly below last year’s levels. Deliveries fell    │
│ 38.6% compared with Q3 2025, when buyers rushed to take advantage of expiring federal incentives and pushed the │
│ market to a record-breaking quarter.                                                                            │
│                                                                                                                 │
│ The latest figures suggest the U.S. EV market is gradually stabilizing after a sharp correction, although the   │
│ industry continues to face challenges as automakers adjust their strategies and consumers reconsider the cost   │
│ of going electric. [...] ### EV Market Share Climbs to 6.5%                                                     │
│                                                                                                                 │
│ Electric vehicles accounted for approximately 6.5% of all new-vehicle sales in the United States during Q3      │
│ 2026, up from 5.8% in the previous quarter. It was the strongest quarterly market share since Q3 2025, when EVs │
│ exceeded 10% of new-vehicle sales for the first time.                                                           │
│                                                                                                                 │
│ When conventional hybrids and plug-in vehicles are included, nearly one in four new vehicles sold during the    │
│ quarter had some form of electrification.                                                                       │
│                                                                                                                 │
│                                                                                                                 │
│ [2] Macron's Final Paris Motor Show Visit Highlights French Auto Industry Record - News and Statistics -        │
│ IndexBox                                                                                                        │
│ URL: https://www.indexbox.io/blog/macrons-final-paris-motor-show-visit-highlights-french-auto-industry-record   │
│ Score: 0.80698997                                     

╭───────────────────────────────────────────── Tavily Search Results ─────────────────────────────────────────────╮
│ Query: EV market growth regulatory changes policy 2025                                                          │
│                                                                                                                 │
│ Results: 5 found                                                                                                │
│ --------------------------------------------------                                                              │
│                                                                                                                 │
│ [1] North America EV Charging Market 2026 | Full Report                                                         │
│ URL:                                                                                                            │
│ https://www.evinfrastructurenews.com/ev-networks/north-america-s-ev-charging-market-in-2026-growth-policy-shift │
│ s-and-key-players                                                                                               │
│ Score: 0.8338803                                                                                                │
│ Content: ## US federal policy changes in 2025-2026                                                              │
│                                                                                                                 │
│ As of 2026, the US$7,500 federal EV tax credits for new EV purchases and the US$4,000 credit for used EVs has   │
│ ended for vehicles acquired after 30th September 2025. The One Big Beautiful Bill Act was signed on 4th July    │
│ 2025 and signalled the end of the EV tax credits in 2026. This includes 30D new EV credits, 25E used EV         │
│ credits, and 45W credits for commercial EVs. [...] Yahoo Finance and Research and Markets estimate the North    │
│ America EV market size will grow from US$95.05 billion in 2025 to an anticipated value of US$102.66 billion in  │
│ 2026 and US$150.92 billion by 2031. This represents a CAGR of 8.01% between 2026-2031, driven by lower battery  │
│ costs, improved EV range, government incentives, and increased interest in domestic battery production (for     │
│ example, in the battery belt region of the US). Passenger cars capture 77.13% of the North American EV market,  │
│ [...] Fortune Business Insights estimates the global EV charging infrastructure market in 2025 was US$45.78     │
│ billion, expected to grow to US$55.97 billion by the end of 2026, and reach US$279.34 billion by 2034 with a    │
│ CAGR of 22.3%.                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
│ [2] Green Eco Tires Market to Reach USD 87.515 Billion by 2035 as Tire-Label Rules, Sustainable Materials and   │
│ EV Efficiency Drive Adoption, Reports Radial Insights                                                           │
│ URL:                                                                                                            │
│ https://www.prnewswire.com/news-releases/green-eco-tires-market-to-reach-usd-87-515-billion-by-2035-as-tire-lab │
│ el-rules-sustainable-materials-and-ev-efficiency-drive-adoption-reports-radial-insights-302903724.html          │
│ Score: 0.4767995                                                                                                │
│ Content: Asia-Pacific Leads Global Market Value: Asia-Pacific holds 44.78% of 2025 demand, supported by         │
│ automotive manufacturing scale, EV production and growing sustainability requirements.                          │
│  Europe Sets the Regulatory Pace: Europe represents 23

...

╭───────────────────────────────────────────── Tavily Search Results ─────────────────────────────────────────────╮
│ Query: EV market concerns challenges enthusiasm drivers 2025 electric vehicle industry outlook                  │
│                                                                                                                 │
│ Results: 10 found                                                                                               │
│ --------------------------------------------------                                                              │
│                                                                                                                 │
│ [1] 2025 US Electric Vehicle Landscape: Market Growth and Regulatory Pressures                                  │
│ URL:                                                                                                            │
│ https://www.morganlewis.com/pubs/2025/12/2025-us-electric-vehicle-landscape-market-growth-and-regulatory-pressu │
│ res                                                                                                             │
│ Score: 0.87218446                                                                                               │
│ Content: December 15, 2025                                                                                      │
│                                                                                                                 │
│ Global sustainability goals, rapid technological innovation, and shifting consumer preferences are driving      │
│ electric vehicle (EV) adoption at an unprecedented pace. Developments in charging infrastructure, energy        │
│ integration, and digital connectivity float amid a constant current of changing regulatory frameworks, market   │
│ dynamics, and competitive pressures. This Insight examines the current outlook for the US EV market, ongoing    │
│ changes to federal and state regulatory policy, critical infrastructure and cybersecurity challenges, and the   │
│ broader trajectory of the industry going forward.                                                               │
│                                                                                                                 │
│ Market Trends and Projections [...] Retail forecasts also reflect this optimism. EVs are expected to account    │
│ for 9.1% of total vehicle sales in 2025, with projections indicating it could rise to 11.8% in 2026 and 26% by  │
│ 2030. This growth is largely driven by expanding model availability, improved battery technology, and           │
│ increasing public awareness. Despite policy and infrastructure hurdles, the market outlook remains positive,    │
│ signaling a steady shift toward electrification in the years to come.                                           │
│                                                                                                                 │
│ Federal and State Policy Dynamics                                                                               │
│                                                                                                                 │
│                                                                                                                 │
│ [2] EY survey: Interest in EVs declines in Singapore | IDNFinancials                                            │
│ URL: https://www.idnfinancials.com/news/60305/ey-survey-interest-in-evs-declines-in-singapore                   │
│ Score: 0.8005209                                                                                                │
│ Content: JAKARTA - Interest among Singaporean drivers in electric vehicles (EVs) is beginning to show signs of  │
│ weakening, with more consumers reconsidering petrol‑powered or internal combustion engine (ICE) cars.           │
│                                                       

╭───────────────────────────────────────────── Tavily Search Results ─────────────────────────────────────────────╮
│ Query: EV industry retail investor sentiment Reddit social media 2025                                           │
│                                                                                                                 │
│ Results: 10 found                                                                                               │
│ --------------------------------------------------                                                              │
│                                                                                                                 │
│ [1] The 25 Most Talked About Stocks on Reddit in 2025 - And What Investors Are Getting Right - AltIndex         │
│ URL: https://altindex.com/news/most-talked-about-reddit-stocks-half-year-checkin                                │
│ Score: 0.71377134                                                                                               │
│ Content: ### Notable Mentions                                                                                   │
│                                                                                                                 │
│ #### 1. Tesla – Still the King of Reddit, But Not the Market                                                    │
│                                                                                                                 │
│ Mentions: 176,677 | Sentiment: Neutral | YTD Change: -26%                                                       │
│                                                                                                                 │
│ Tesla remains the most talked-about stock on Reddit by a wide margin. Despite the ongoing chatter, the stock is │
│ down significantly this year. The conversation has shifted from bullish enthusiasm to more mixed debates about  │
│ slowing growth, the feud between Elon and Trump, management decisions, Robotaxi launch, and EV competition.     │
│ This is a clear case where high Reddit volume doesn’t always translate to positive price action.                │
│                                                                                                                 │
│ #### 2. GameStop – Meme Stock Energy Isn’t Dead                                                                 │
│                                                                                                                 │
│ Mentions: 113,624 | Sentiment: Bullish | YTD Change: -24% [...] Mentions: 113,624 | Sentiment: Bullish | YTD    │
│ Change: -24%                                                                                                    │
│                                                                                                                 │
│ GameStop continues to ignite the retail crowd, especially during short squeeze rumors. Reddit sentiment is      │
│ still strong, but the stock is struggling fundamentally. While many see it as a cult favorite, its performance  │
│ shows the disconnect that can exist between community excitement and business reality.                          │
│                                                                                                                 │
│ #### 3. NVIDIA – A Beloved AI Bet                                                                               │
│                                                                                                                 │
│ Mentions: 96,326 | Sentiment: Bullish | YTD Change: +14%                                                        │
│                                                                                                                 │
│ NVIDIA is one of the many names where bullish Reddit sentiment aligns with solid returns. The AI narrative      │
│ remains strong, and retail investors are fully bought 

╭───────────────────────────────────────────── Tavily Search Results ─────────────────────────────────────────────╮
│ Query: electric vehicle stocks market sentiment investor opinions 2025                                          │
│                                                                                                                 │
│ Results: 10 found                                                                                               │
│ --------------------------------------------------                                                              │
│                                                                                                                 │
│ [1] 10 Best Electric Vehicle Stocks for 2026 and How to Invest | The Motley Fool                                │
│ URL:                                                                                                            │
│ https://www.fool.com/investing/stock-market/market-sectors/consumer-discretionary/automotive-stocks/electric-ca │
│ r-stocks                                                                                                        │
│ Score: 0.8893851                                                                                                │
│ Content: ### 4. General Motors                                                                                  │
│                                                                                                                 │
│ Legacy automaker GM may not be a name that investors immediately recognize as an EV leader, but it's building   │
│ momentum. Through the first half of 2026, GM had the second-highest EV sales in the United States.              │
│                                                                                                                 │
│ Paul Jacobson, the company's CFO, expressed optimism about EVs on GM's Q4 2025 conference call, stating, "As    │
│ consumer adoption of EVs increases, albeit at a slower pace than previously anticipated, we expect to achieve   │
│ the necessary scale to deliver EVs profitably over time."                                                       │
│                                                                                                                 │
│ Dedicated to rewarding shareholders, GM pays a modest dividend while also reinvesting in the company. As of     │
│ August 2026, GM stock offered investors a 0.82% forward dividend yield.                                         │
│                                                                                                                 │
│ ### 5. BYD [...] ### 5. BYD                                                                                     │
│                                                                                                                 │
│ Li Auto represents another Chinese EV leader. As of July 31, 2026, the company had 490 retail stores and 536    │
│ servicing centers. Li Auto also helps customers keep their (electric) motors running, with 4,141 supercharging  │
│ stations equipped with 22,841 charging stalls.                                                                  │
│                                                                                                                 │
│ While profitability has been inconsistent, Li Auto did report net income of $162.9 million in 2025. Its sizable │
│ cash position of $13.7 billion (as of March 31, 2026) makes Li Auto well-positioned to withstand temporary      │
│ market downturns while also allocating resources to reinvest in the business.                                   │
│                                                                                                                 │
│ ### 7. Volkswagen AG                                                                                            │
│                                                       

╭───────────────────────────────────────────── Tavily Search Results ─────────────────────────────────────────────╮
│ Query: EV electric vehicle sector outlook 2025 market trends growth                                             │
│                                                                                                                 │
│ Results: 10 found                                                                                               │
│ --------------------------------------------------                                                              │
│                                                                                                                 │
│ [1] 2025 US Electric Vehicle Landscape: Market Growth and Regulatory Pressures                                  │
│ URL:                                                                                                            │
│ https://www.morganlewis.com/pubs/2025/12/2025-us-electric-vehicle-landscape-market-growth-and-regulatory-pressu │
│ res                                                                                                             │
│ Score: 0.93885136                                                                                               │
│ Content: Retail forecasts also reflect this optimism. EVs are expected to account for 9.1% of total vehicle     │
│ sales in 2025, with projections indicating it could rise to 11.8% in 2026 and 26% by 2030. This growth is       │
│ largely driven by expanding model availability, improved battery technology, and increasing public awareness.   │
│ Despite policy and infrastructure hurdles, the market outlook remains positive, signaling a steady shift toward │
│ electrification in the years to come.                                                                           │
│                                                                                                                 │
│ Federal and State Policy Dynamics [...] Market Trends and Projections                                           │
│                                                                                                                 │
│ Since 2020, EV sales have surged from just over 500,000 to 1.6 million in 2024. As 2025 draws to a close, the   │
│ US EV market has remained on a strong upward trend, with a projected 2.25 million sales projected by year’s     │
│ end. While this past October saw sales dip following the expiration of federal tax credits, quarterly growth    │
│ remains robust, with Q3 2025 sales up 40% year-over-year. Analysts expect this slowdown to be temporary, with   │
│ long-term demand being supported by resilient consumer interest and innovation. [...] December 15, 2025         │
│                                                                                                                 │
│ Global sustainability goals, rapid technological innovation, and shifting consumer preferences are driving      │
│ electric vehicle (EV) adoption at an unprecedented pace. Developments in charging infrastructure, energy        │
│ integration, and digital connectivity float amid a constant current of changing regulatory frameworks, market   │
│ dynamics, and competitive pressures. This Insight examines the current outlook for the US EV market, ongoing    │
│ changes to federal and state regulatory policy, critical infrastructure and cybersecurity challenges, and the   │
│ broader trajectory of the industry going forward.                                                               │
│                                                                                                                 │
│ Market Trends and Projections                                                                                   │
│                                                                                                                 │
│                                                       

# ⚡ EV Sector Investment Analysis: A Comprehensive Overview

Here's a synthesized breakdown from all three of my specialist agents to help you think through the EV investment landscape.

---

## 🌍 The Big Picture

The EV industry is a **genuine long-term growth story** — global sales hit ~20.7 million units in 2025 and are projected to reach 35+ million by 2030. However, **the near-term environment is choppy**, particularly in the U.S., where federal EV tax credits were eliminated in mid-2025, causing notable demand headwinds.

---

## 📰 Key Developments (News)

| Factor | Detail |
|---|---|
| **U.S. Policy Headwind** | Federal $7,500 EV tax credit eliminated after Sept. 2025 — significant blow to near-term demand |
| **Europe Accelerating** | BEV market share hit 33% in Europe; Chinese brands aggressively expanding |
| **Tesla Still Dominates** | ~54% U.S. EV market share, but sales volumes are declining |
| **Rivian Bright Spot** | +46% YoY sales growth; R2 model and commercial vans 

## Try different questions

In [11]:
questions = [
    "What's the outlook for renewable energy stocks?",
    "Is the semiconductor industry a good investment right now?",
]

for q in questions:
    print(f"\n{'=' * 80}\nQuestion: {q}\n{'=' * 80}")
    supervisor(q)


Question: What's the outlook for renewable energy stocks?
Great follow-up question! Renewable energy has strong thematic overlap with EVs but its own distinct dynamics. Let me fire all three specialist agents simultaneously to get a full picture!
Tool #4: news_agent

Tool #5: financial_agent

Tool #6: sentiment_agent


DEPRECATION WARNING: tavily is deprecated. This warning becomes an error log in v0.9.0. Migration path: use the official Tavily MCP server, documented at https://docs.tavily.com/documentation/mcp.
DEPRECATION WARNING: tavily is deprecated. This warning becomes an error log in v0.9.0. Migration path: use the official Tavily MCP server, documented at https://docs.tavily.com/documentation/mcp.
DEPRECATION WARNING: tavily is deprecated. This warning becomes an error log in v0.9.0. Migration path: use the official Tavily MCP server, documented at https://docs.tavily.com/documentation/mcp.
DEPRECATION WARNING: tavily is deprecated. This warning becomes an error log in v0.9.0. Migration path: use the official Tavily MCP server, documented at https://docs.tavily.com/documentation/mcp.
DEPRECATION WARNING: tavily is deprecated. This warning becomes an error log in v0.9.0. Migration path: use the official Tavily MCP server, documented at https://docs.tavily.com/documentation/mcp.
DEPRECATION WAR

╭───────────────────────────────────────────── Tavily Search Results ─────────────────────────────────────────────╮
│ Query: Orsted offshore wind energy news 2025                                                                    │
│                                                                                                                 │
│ Results: 7 found                                                                                                │
│ --------------------------------------------------                                                              │
│                                                                                                                 │
│ [1] US Gets New Offshore Wind Farm as All Turbines Installed at 704 MW Revolution Wind - Offshore Wind          │
│ URL:                                                                                                            │
│ https://www.offshorewind.biz/2026/09/18/us-gets-new-offshore-wind-farm-as-all-turbines-installed-at-704-mw-revo │
│ lution-wind                                                                                                     │
│ Score: 0.8565368                                                                                                │
│ Content: On 22 August 2025, the US Department of the Interior’s Bureau of Ocean Energy Management (BOEM) issued │
│ a stop-work order covering offshore construction at Revolution Wind. Ørsted and Skyborn Renewables filed a      │
│ lawsuit challenging the order, and a federal court granted a preliminary injunction on 22 September, allowing   │
│ affected construction activities to resume. [...] By August 2025, all foundations and 45 turbines had been      │
│ installed, and the project was approximately 80 per cent complete. Installation was affected by damage to a leg │
│ of one of the installation vessels in June 2025, which required five weeks of inspection and repairs. A delay   │
│ in construction of the onshore substation had also pushed the expected commercial operation date from 2025 to   │
│ the second half of 2026. [...] A second federal stop-work order issued on 22 December 2025 paused construction  │
│ again, with seven turbines remaining to be installed at the time. On 12 January 2026, the US District Court for │
│ the District of Columbia granted the project’s request for a preliminary injunction, allowing construction to   │
│ restart while the legal proceedings continued.                                                                  │
│                                                                                                                 │
│                                                                                                                 │
│ [2] Ørsted and Nuveen inaugurate Borkum Riffgrund 3, one of Germany’s largest offshore wind farms               │
│ URL: https://orsted.com/en/media/news/2026/09/orsted-and-nuveen-inaugurate-borkum-riffgrund-3-on-15173591       │
│ Score: 0.8312667                                                                                                │
│ Content: More broadly, offshore wind creates value well beyond coastal regions, including in industrial regions │
│ such as North Rhine-Westphalia, Baden-Württemberg, and Saarland. In 2025, the offshore wind industry generated  │
│ more than 30,000 full-time jobs and EUR 14.6 billion in gross value across Germany, underlining the sector’s    │
│ role as both an energy and an industrial asset. [...] Widely recognised as a global sustainability leader,      │
│ Ørsted is guided by its vision of a world that runs entirely on green energy. Headquartered in Denmark, Ørsted  │
│ employs approximately 7,300 people. Ørsted's shares are listed on Nasdaq Copenhagen (Orsted). In 2025, the      │
│ group's operating profit excluding new partnerships and cancellation fees was DKK 25.1 billion (EUR 3.4         │
│ billion). [...] Nuveen is a global investment leader, 

╭───────────────────────────────────────────── Tavily Search Results ─────────────────────────────────────────────╮
│ Query: First Solar Enphase SunPower solar energy news 2025                                                      │
│                                                                                                                 │
│ Results: 4 found                                                                                                │
│ --------------------------------------------------                                                              │
│                                                                                                                 │
│ [1] U.S. Solar Manufacturing Capex Forecast to Fall in 2027 - News and Statistics - IndexBox                    │
│ URL:                                                                                                            │
│ https://www.indexbox.io/blog/us-solar-manufacturing-capex-to-decline-in-2027-as-over-70-of-announced-projects-s │
│ tall                                                                                                            │
│ Score: 0.4808937                                                                                                │
│ Content: News…                                                                                                  │
│                                                                                                                 │
│ #1                                                                                                              │
│                                                                                                                 │
│ F                                                                                                               │
│                                                                                                                 │
│ ### First Solar, Inc.                                                                                           │
│                                                                                                                 │
│ Headquarters                                                                                                    │
│                                                                                                                 │
│ Tempe, Arizona                                                                                                  │
│                                                                                                                 │
│ Focus                                                                                                           │
│                                                                                                                 │
│ Thin-film cadmium telluride (CdTe) solar modules                                                                │
│                                                                                                                 │
│ Scale                                                                                                           │
│                                                                                                                 │
│ Large-scale manufacturer                                                                                        │
│                                                                                                                 │
│ Leading US solar module producer; vertically integrated                                                         │
│                                                                                                                 │
│ #2                                                                                                              │
│                                                       

╭───────────────────────────────────────────── Tavily Search Results ─────────────────────────────────────────────╮
│ Query: renewable energy sector latest news 2025                                                                 │
│                                                                                                                 │
│ Results: 7 found                                                                                                │
│ --------------------------------------------------                                                              │
│                                                                                                                 │
│ [1] Shock Therapy in the Land of Dragons: A Clean Energy Transition in Vietnam and Its Energy Security          │
│ Implications - The SAIS Review of International Affairs                                                         │
│ URL:                                                                                                            │
│ https://saisreview.sais.jhu.edu/shock-therapy-in-the-land-of-dragons-a-clean-energy-transition-in-vietnam-and-i │
│ ts-energy-security-implications                                                                                 │
│ Score: 0.81770587                                                                                               │
│ Content: Over time, PDP8 and its energy transition goals for Vietnam’s electricity sector have been repeatedly  │
│ updated to be more ambitious. In the most recent iteration from April 15, 2025, the targets are breathtaking:   │
│ non-hydropower renewable energy technologies are to produce 28-36% of electricity by 2030 and 74-75% by 2050.   │
│ Furthermore, the expansive buildout of wind and solar power mandated by PDP8 runs concurrently with plans to    │
│ phase out the use of coal for electricity production entirely (the [...] “Viet Nam – Countries & Regions –      │
│ Coal,” _International Energy Agency,_ accessed December 12, 2025,                                               │
│                                                                                                                 │
│ “Vietnam’s 15 Comprehensive Strategic Partners,” _Vietnam Law & Legal Forum_, February 4, 2026,                 │
│                                                                                                                 │
│ Hanh Nguyen Vu, “Vietnam’s Economy in 2025: GDP, FDI, and Trade,” _Vietnam Briefing News_, February 4, 2026,    │
│                                                                                                                 │
│ Hanh Nguyen Vu, “Vietnam Revises PDP8: Targets of the National Power Development Plan,” _Vietnam Briefing_      │
│ _News_, September 26, 2025,                                                                                     │
│                                                                                                                 │
│ FacebookMastodonEmailShare [...] Aditya Saroha et al., “Vietnam Emerges as Key Demand Hub as Top Buyers Cut     │
│ Back on Thermal Coal Imports,” _S&P Global Energy_, June 4, 2025,                                               │
│                                                                                                                 │
│ Sudhanshu Singh, “Vietnam’s Power Shortage: Impacts and Government Response,” _Vietnam Briefing,_ September 3,  │
│ 2025,                                                                                                           │
│                                                                                                                 │
│ Carlyle A. Thayer, “Vietnam’s Strategy of ‘Cooperating and Struggling’ with China over Maritime Disputes in the │
│ South China Sea,” _Journal of Asian Security and International Affairs_ (New Delhi, India) 3, no. 2 (2016):     │
│ 200–220,                                              

...

╭───────────────────────────────────────────── Tavily Search Results ─────────────────────────────────────────────╮
│ Query: Intel semiconductor turnaround strategy foundry 2025 2026                                                │
│                                                                                                                 │
│ Results: 5 found                                                                                                │
│ --------------------------------------------------                                                              │
│                                                                                                                 │
│ [1] Intel vs. SK Hynix: Which Semiconductor Stock Is a Better Buy in 2026? - The Globe and Mail                 │
│ URL:                                                                                                            │
│ https://www.theglobeandmail.com/investing/markets/stocks/SKHY/pressreleases/5271355/intel-vs-sk-hynix-which-sem │
│ iconductor-stock-is-a-better-buy-in-2026                                                                        │
│ Score: 0.82761955                                                                                               │
│ Content: Intel operates as an integrated device manufacturer, designing and producing processors and            │
│ accelerators for cloud and edge computing. The company is currently undergoing a strategic transformation to    │
│ expand its foundry services, which involves building chips for other designers. This strategy includes          │
│ significant operational ties with the U.S. government, which held a 10% equity stake in the organization as of  │
│ March 2026. Customer concentration like this adds a layer of risk to the business, [...] Intel is pressing on   │
│ with its plan to revive its business. In the spring it announced a partnership with Space Exploration           │
│ Technologies Inc. (NASDAQ:SPCX) and Tesla Inc. (NASDAQ:TSLA) to build Terafab, a massive AI chip-making plant   │
│ aiming to produce as many as 200 billion AI chips a year. Intel is providing its foundry expertise and its 14A  │
│ process, an effort to produce 1.4-nanometer chips it announced two years ago and is still developing. [...] All │
│ that has generated some optimism on Wall Street. For fiscal 2026, sales are seen rising 19% to $62.3 billion,   │
│ though a large $11 billion net loss is projected, reflecting the cost of expanding its U.S. foundries.          │
│                                                                                                                 │
│                                                                                                                 │
│ [2] Intel Drops 4% as Elon Musk Signals Taiwan Semiconductor Could Join Terafab; AMD Slips, NVIDIA Holds Steady │
│ - 24/7 Wall St.                                                                                                 │
│ URL:                                                                                                            │
│ https://247wallst.com/investing/2026/10/05/intel-drops-4-as-elon-musk-signals-taiwan-semiconductor-could-join-t │
│ erafab-amd-slips-nvidia-holds-steady                                                                            │
│ Score: 0.7140107                                                                                                │
│ Content: Terafab’s planned output is reserved for Musk’s own companies, which made Intel’s role there a         │
│ showcase for its manufacturing ambitions and a signal to other potential foundry customers, and a Taiwan        │
│ Semiconductor role would place an established contract manufacturer alongside Intel at the project Intel backed │
│ first. That shift carries weight because Intel’s external foundry revenue came to just $293 million in the      │
│ second quarter of 2026, leaving few outside wins as vi

╭───────────────────────────────────────────── Tavily Search Results ─────────────────────────────────────────────╮
│ Query: ASML extreme ultraviolet lithography orders demand outlook 2025 2026                                     │
│                                                                                                                 │
│ Results: 5 found                                                                                                │
│ --------------------------------------------------                                                              │
│                                                                                                                 │
│ [1] ASML Q3 Earnings Preview: Revenue May Rise 55% YoY, Wall Street Sees Up to $2,866                           │
│ URL: https://www.tradingkey.com/analysis/stocks/us-stocks/262211059-asml-q3-earnings-preview-revenue-tradingkey │
│ Score: 0.91156644                                                                                               │
│ Content: ASML is projected to report strong third-quarter fiscal 2026 financial results on October 14, with     │
│ expected revenue of approximately €11.6 to €11.8 billion, driven by robust AI-related demand for EUV            │
│ lithography equipment. While anticipated growth is largely priced in, future stock performance will depend on   │
│ gross margins reaching the upper end of guidance, resilient service revenue, and strong net bookings. Long-term │
│ prospects remain supported by high-NA EUV adoption, robust customer capital [...] AI-generated summary          │
│                                                                                                                 │
│ TradingKey - ASML (ASML) will report its third-quarter fiscal 2026 financial results on October 14. As a core   │
│ global supplier of extreme ultraviolet (EUV) lithography equipment, the company is benefiting from demand for   │
│ advanced chips driven by the expansion of artificial intelligence infrastructure. [...] The company previously  │
│ stated that its low-NA EUV capacity would be about 65 units in 2026, with plans to increase it by about 30% in  │
│ 2027 and explore a further 30% expansion in 2028. DUV immersion lithography machines are following a similar    │
│ expansion path, with capacity in 2026 at around 130 units. Management stated that 2027 low-NA EUV capacity is   │
│ nearly fully covered by customer orders, and significant purchasing commitments have also been secured for      │
│ 2028.                                                                                                           │
│                                                                                                                 │
│                                                                                                                 │
│ [2] ASML: Memory Shortages Fuel a New Lithography Boom                                                          │
│ URL: https://drrobertcastellano.substack.com/p/asml-memory-shortages-fuel-a-new                                 │
│ Score: 0.87807417                                                                                               │
│ Content: ASML is particularly well positioned to benefit because its extreme ultraviolet lithography systems    │
│ are essential for advanced logic manufacturing and increasingly important for leading-edge DRAM production. Its │
│ deep ultraviolet systems also remain necessary across multiple semiconductor process layers. The company raised │
│ its full-year 2026 revenue guidance in July to €43 billion–€45 billion, establishing a considerably higher      │
│ benchmark for its upcoming earnings report. Investors must now [...] According to Chart 1, The Street           │
│ anticipates substantial second-half 2026 equipment revenue growth across the four major semiconductor equipment │
│ manufacturers. KLA (KLAC) is projected to increase rev

╭───────────────────────────────────────────── Tavily Search Results ─────────────────────────────────────────────╮
│ Query: Broadcom Qualcomm AMD AI chip custom silicon news 2025 2026                                              │
│                                                                                                                 │
│ Results: 5 found                                                                                                │
│ --------------------------------------------------                                                              │
│                                                                                                                 │
│ [1] Qualcomm-AWS $60B AI Chip Deal: Stock Jumps 7% [2026]                                                       │
│ URL: https://tech-insider.org/qualcomm-aws-60-billion-ai-chip-deal-2026                                         │
│ Score: 0.8719229                                                                                                │
│ Content: Qualcomm is a late arrival to a custom AI chip market that Broadcom and Marvell have spent the past    │
│ two years building into a serious Nvidia alternative. Broadcom’s OpenAI partnership, announced October 13,      │
│ 2025, covers 10 gigawatts of custom accelerators in a deal Reuters and the Wall Street Journal both described   │
│ as worth roughly $10 billion, with initial chips (code-named Jalapeño) targeted for deployment by the end of    │
│ 2026. Broadcom separately signed a long-term agreement with Google on [...] No single custom silicon deal       │
│ threatens Nvidia’s near-term dominance. Even generous estimates of the custom ASIC market, in the $35 billion   │
│ to $70 billion range for 2026, remain a fraction of the broader AI accelerator market that multiple analyst     │
│ firms peg above $150 billion this year, with Nvidia still capturing the large majority of that revenue. But the │
│ cumulative effect of Broadcom’s OpenAI and Google deals, Marvell’s AWS and Microsoft relationships, and now     │
│ Qualcomm’s Meta, Microsoft, and [...] Broadcom’s custom silicon business already includes substantial           │
│ networking and optical DSP revenue alongside its accelerator work.                                              │
│                                                                                                                 │
│                                                                                                                 │
│ [2] Synopsys Signs $1B+ Amazon Silicon IP Deal [2026]                                                           │
│ URL: https://tech-insider.org/synopsys-amazon-1-billion-silicon-ip-deal-2026                                    │
│ Score: 0.82186085                                                                                               │
│ Content: The Synopsys-Amazon agreement is the latest in a string of billion-dollar-plus custom-silicon          │
│ commitments across the cloud and chip industry in 2026. Qualcomm and AWS signed their own multibillion-dollar   │
│ AI chip arrangement earlier this year, and competing cloud providers have pursued parallel paths with their own │
│ custom CPU lines, as seen in the ongoing race between AWS Graviton5, Azure Cobalt 200 and Google Axion. The     │
│ common thread across all of these deals is the same: hyperscalers want [...] The deal also sits inside a larger │
│ shift the industry has been tracking all year: AWS’s biggest GPU supplier, Nvidia, is increasingly competing    │
│ with its own largest customers as they build custom silicon in-house. Broadcom and Marvell, which both design   │
│ custom AI accelerators for hyperscalers, occupy a similar space to Synopsys in the sense that all three         │
│ companies profit from the same trend — cloud providers wanting workload-specific chips instead of off-the-shelf │
│ GPUs — even though their [...] Is this Amazon’s first 

╭───────────────────────────────────────────── Tavily Search Results ─────────────────────────────────────────────╮
│ Query: Micron memory chip HBM AI demand 2025 2026                                                               │
│                                                                                                                 │
│ Results: 5 found                                                                                                │
│ --------------------------------------------------                                                              │
│                                                                                                                 │
│ [1] Beyond the Memory Cycle: AI, HBM, and the New Semiconductor Shortage                                        │
│ URL: https://www.csis.org/analysis/beyond-memory-cycle-ai-hbm-and-new-semiconductor-shortage                    │
│ Score: 0.8815438                                                                                                │
│ Content: New capacity: New memory capacity has been coming online in 2026, but it is unlikely to alleviate the  │
│ increasing supply constraints. Despite several major expansion projects, industry forecasts suggest that memory │
│ manufacturers will only meet around 60 percent of global demand in calendar year 2026, reflecting the           │
│ extraordinary pace of AI-driven consumption.                                                                    │
│        In February 2026, Micron began operations at its $2.75 billion DRAM and NAND assembly and test facility  │
│ in Gujarat, India. [...] As semiconductor analyst Claus Aasholm notes, AI demand is highly price inelastic,     │
│ creating powerful incentives for memory manufacturers to maximize HBM production. Industry leaders increasingly │
│ believe this represents a structural rather than cyclical transformation. In February 2026, Lam Research CEO    │
│ Tim Archer argued that AI-driven memory demand would “overwhelm all other sources of demand” through the end of │
│ the decade. [...] Market dynamics mean that HBM used for AI chips by companies such as Nvidia and Advanced      │
│ Micro Devices affects production capacity for commodity DRAM used in smartphones, PCs, automobiles, and other   │
│ consumer electronics. As the AI supercycle continues, data centers are projected to consume approximately 70    │
│ percent of all memory devices produced worldwide in 2026, underscoring how AI is reshaping both the composition │
│ of memory production and the allocation of industry investment.                                                 │
│                                                                                                                 │
│                                                                                                                 │
│ [2] 4 Memory Semiconductor Stocks to Watch in October 2026                                                      │
│ URL: https://finance.yahoo.com/markets/stocks/articles/4-memory-semiconductor-stocks-watch-131800867.html       │
│ Score: 0.767638                                                                                                 │
│ Content: Western Digital Corporation price-consensus-chart | Western Digital Corporation Quote                  │
│                                                                                                                 │
│ Micron Technology is the key beneficiary of the AI-driven memory and storage market. Tighter DRAM and NAND      │
│ supply and a richer mix of HBM, data center SSDs and high-capacity products are driving MU's business. Micron   │
│ Technology is reinforcing its competitive position through long-term strategic customer agreements, which       │
│ secure multi-year demand across DRAM, HBM and NAND products. [...] Given the high demand for HBM3E, HBM4 and    │
│ future generations exceeds the company's ability to su

# 🔬 Semiconductor Industry: Comprehensive Investment Analysis

Here's a fully synthesized picture across news, financials, and sentiment for the chip sector — arguably the most consequential investment theme in the global economy right now.

---

## 🌍 The Big Picture

The semiconductor industry is in a **historic AI-driven supercycle**. Global data center chip spending hit $314.87 billion in 2025 and is projected to surpass **$1 trillion by 2035**. The Philadelphia Semiconductor Index (SOX) is up **62.1% year-to-date**, and hyperscalers (Microsoft, Google, Amazon, Meta) are projected to spend **~$950 billion+ on AI infrastructure in 2027** alone. Yet underneath the euphoria, serious questions about sustainability, geopolitics, and valuations are creating growing anxiety.

---

## 📰 Key Developments (News)

| Factor | Detail |
|---|---|
| **AI Supercycle** | AI accelerators now represent 54% of all data center chip spending |
| **NVIDIA's Dominance** | ~$5.67T market cap; SpaceX renting

## Key takeaways

1. **Specialisation** — each worker has one job and the right tools for it
2. **Coordination** — the supervisor decides who to ask and combines the answers
3. **Modularity** — add a new worker without touching the others

**Make it production-ready:** retries and caching for API calls, rate limits, async execution, and tracing (Section 7).